# Imports

In [ ]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [ ]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
api.start_spark(n_executors=400, config=config)

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [ ]:
%run ./shared_variables.ipynb
%run ./S3_csv_writers.ipynb

In [ ]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [ ]:
# # write a Spark DataFrame to your S3 user space
# from tdp_spark import spark_connect_tdp, spark_write_cre, spark_read_cre

# def write_csv(df, path, **kwargs):
#     return df.coalesce(1).write.option("header", True).csv(path, **kwargs)

# def write_csv_with_partitions(df, path, **kwargs):
#     partitions = kwargs.pop("partitionBy")
#     print("partitions:", partitions)
#     return df.write.option("header", True).partitionBy(partitions).csv(path, **kwargs)

In [ ]:
df_AA_raw = (
    api.dataframe("ArincAirport", **dates, metadata=True)
    .select(
        F.col("identification.name").alias("icao_code"),
        F.col("identification.icao_region").alias("icao_region"),
       "full_name",
        "latitude",
        "longitude",
        "elevation",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("full_name", F.regexp_replace("full_name", ",", ""))
    .orderBy("icao_code", "icao_region")
)

In [ ]:
window = Window.partitionBy("icao_code").orderBy(col("end_date").desc())

df_AA = (df_AA_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .drop("end_date")
)

In [ ]:
df_AA.show()

In [ ]:
df_AA.count()

In [ ]:
df_AC_raw = (
    api.dataframe("AirportCodes", **dates, metadata=True)
    .select(
       "icao_code",
       "faa_code",
       "iata_code",
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .orderBy("icao_code")
)

In [ ]:
window = Window.partitionBy("icao_code").orderBy(col("end_date").desc())

df_AC = (df_AC_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
    .drop("end_date")
)

In [ ]:
df_AC.show()

In [ ]:
df_AC.count()

In [ ]:
df_airports = df_AA.join(df_AC, df_AA.icao_code == df_AC.icao_code, how="left").drop(df_AC.icao_code)

In [ ]:
df_airports.count()

In [ ]:
df_airports_output = (
    df_airports
        .select(
            "icao_code", 
            "icao_region", 
            "iata_code", 
            "faa_code", 
            "full_name", 
            "latitude", 
            "longitude", 
            "elevation", 
            "magnetic_variation")
        .orderBy("icao_code", "iata_code", "faa_code", "icao_region")
)

In [ ]:
df_airports_output.show()

In [ ]:
# write a Spark DataFrame to your HDFS space
(
    df_airports_output
    .repartition(1)
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/airports", compression="None", mode="overwrite")
)

In [ ]:
# write a Spark DataFrame to your S3 user space
#spark_write_cre(df_airports_output, path="CRAFT/" + year0 + "/airports", writer=write_csv, compression=None, mode="overwrite")